# Phase 4–10: BLAIR Encoding, Classifiers, Pipeline & Evaluation

**Project:** Review-Grounded NLP System for Product Recommendation and Review Summarization in Beauty Products

**Team:** Anushka Thagle (axt5884) | Supanut Chindawan (sxc6473)

**Platform:** Google Colab Pro (T4 / A100 GPU)

**Prerequisite:** Phase 0–3 must be completed. CNN model and processed data should be in Google Drive.

---

| Phase | Description | Estimated Time |
|-------|-------------|----------------|
| 4 | BLAIR Encoding + FAISS Vector Database | 4–6 hours |
| 5 | BiLSTM Clarification Classifier | 30–60 min |
| 6 | BiLSTM Extractive Summarizer | 1–2 hours |
| 7 | Ground Truth Construction + Retrieval Eval | 1–2 hours |
| 8 | End-to-End Pipeline | 30 min |
| 9 | Faithfulness Evaluation | 30 min |
| 10 | Results Summary & Demo | 30 min |

---
## Setup: Mount Drive & Install Packages

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/NLP_Project"

import os
DIRS = {
    "data_raw":   f"{PROJECT_ROOT}/data/raw",
    "data_proc":  f"{PROJECT_ROOT}/data/processed",
    "models":     f"{PROJECT_ROOT}/models",
    "results":    f"{PROJECT_ROOT}/results",
    "embeddings": f"{PROJECT_ROOT}/data/embeddings",
    "faiss":      f"{PROJECT_ROOT}/data/faiss",
    "ground_truth": f"{PROJECT_ROOT}/data/ground_truth",
}
for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"  {name:>14}: {path}")

Mounted at /content/drive
        data_raw: /content/drive/MyDrive/NLP_Project/data/raw
       data_proc: /content/drive/MyDrive/NLP_Project/data/processed
          models: /content/drive/MyDrive/NLP_Project/models
         results: /content/drive/MyDrive/NLP_Project/results
      embeddings: /content/drive/MyDrive/NLP_Project/data/embeddings
           faiss: /content/drive/MyDrive/NLP_Project/data/faiss
    ground_truth: /content/drive/MyDrive/NLP_Project/data/ground_truth


In [3]:
%%capture
!pip install datasets==2.17.0
!pip install transformers
!pip install faiss-cpu
!pip install spacy
!pip install rouge-score
!python -m spacy download en_core_web_sm

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import json
import random
import time
import faiss
import pickle
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, f1_score, precision_score,
    recall_score, confusion_matrix, ndcg_score
)
import matplotlib.pyplot as plt

# Reproducibility
SEED = 84
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda:0
GPU: NVIDIA A100-SXM4-40GB


### Load Data from Phase 0–3

In [5]:
def load_jsonl(filepath):
    """Load a JSONL file into a list of dicts."""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

# Load labeled sentences from Phase 2
LABELED_FILE = os.path.join(DIRS["data_proc"], "sentences_labeled.parquet")
df_sent_labeled = pd.read_parquet(LABELED_FILE)
print(f"Labeled sentences: {len(df_sent_labeled):,}")

# Load all sentences (including 3-star)
SENT_FILE = os.path.join(DIRS["data_proc"], "sentences.jsonl")
df_all_sentences = pd.DataFrame(load_jsonl(SENT_FILE))
print(f"All sentences: {len(df_all_sentences):,}")

# Load metadata
METADATA_FILE = os.path.join(DIRS["data_raw"], "All_Beauty_metadata.jsonl")
metadata_raw = load_jsonl(METADATA_FILE)
df_metadata = pd.DataFrame(metadata_raw)
print(f"Products: {len(df_metadata):,}")

Labeled sentences: 1,825,333
All sentences: 1,991,664
Products: 112,590


---
## Phase 4: BLAIR Encoding + FAISS Vector Database

### Step 11: Load BLAIR Pretrained Model

In [6]:
from transformers import AutoTokenizer, AutoModel

BLAIR_MODEL_NAME = "hyp1231/blair-roberta-base"
BLAIR_DIM = 768

print("Loading BLAIR tokenizer and model...")
blair_tokenizer = AutoTokenizer.from_pretrained(BLAIR_MODEL_NAME)
blair_model     = AutoModel.from_pretrained(BLAIR_MODEL_NAME).to(DEVICE)
blair_model.eval()

print(f"BLAIR loaded: {BLAIR_MODEL_NAME}")
print(f"Embedding dimension: {BLAIR_DIM}")


def encode_texts_blair(texts, batch_size=32, show_progress=True):
    """
    Encode a list of texts into BLAIR embeddings (768-dim).
    Uses FP16 for memory efficiency on GPU.
    Returns: numpy array of shape (len(texts), 768)
    """
    all_embeddings = []
    iterator = range(0, len(texts), batch_size)
    if show_progress:
        iterator = tqdm(iterator, desc="Encoding", total=(len(texts) + batch_size - 1) // batch_size)

    with torch.no_grad(), torch.cuda.amp.autocast():  # FP16
        for i in iterator:
            batch = texts[i:i + batch_size]
            encoded = blair_tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(DEVICE)

            outputs = blair_model(**encoded)
            # Use CLS token as sentence embedding
            embeddings = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(embeddings.float().cpu().numpy())

    return np.vstack(all_embeddings)

Loading BLAIR tokenizer and model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BLAIR loaded: hyp1231/blair-roberta-base
Embedding dimension: 768


### Step 12: Encode Product Metadata

In [7]:
PRODUCT_VECTORS_FILE = os.path.join(DIRS["faiss"], "product_vectors.npy")
PRODUCT_META_FILE    = os.path.join(DIRS["faiss"], "product_metadata.pkl")

if not os.path.exists(PRODUCT_VECTORS_FILE):
    print("Encoding product metadata with BLAIR...")

    # Combine metadata fields into a single text per product
    def build_product_text(row):
        """Concatenate title + features + description + category into one string."""
        parts = []
        if isinstance(row.get("title"), str):
            parts.append(row["title"])
        if isinstance(row.get("features"), list):
            parts.extend([f for f in row["features"] if isinstance(f, str)])
        if isinstance(row.get("description"), list):
            parts.extend([d for d in row["description"] if isinstance(d, str)])
        if isinstance(row.get("main_category"), str):
            parts.append(row["main_category"])
        text = " ".join(parts).strip()
        return text if len(text) > 0 else "beauty product"

    product_texts = df_metadata.apply(build_product_text, axis=1).tolist()
    product_ids   = df_metadata["parent_asin"].tolist()

    print(f"  Products to encode: {len(product_texts):,}")

    # Encode with BLAIR (batch_size=32 for product texts which can be long)
    product_vectors = encode_texts_blair(product_texts, batch_size=16)

    # Normalize vectors for cosine similarity
    norms = np.linalg.norm(product_vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1
    product_vectors = product_vectors / norms

    # Save vectors and metadata
    np.save(PRODUCT_VECTORS_FILE, product_vectors)

    product_meta = {
        "product_ids": product_ids,
        "titles":      df_metadata["title"].tolist(),
        "categories":  df_metadata["main_category"].tolist(),
        "prices":      df_metadata["price"].tolist(),
        "avg_ratings": df_metadata["average_rating"].tolist(),
    }
    with open(PRODUCT_META_FILE, "wb") as f:
        pickle.dump(product_meta, f)

    print(f"  Product vectors saved: {product_vectors.shape}")
else:
    print("Loading cached product vectors...")
    product_vectors = np.load(PRODUCT_VECTORS_FILE)
    with open(PRODUCT_META_FILE, "rb") as f:
        product_meta = pickle.load(f)
    print(f"  Product vectors loaded: {product_vectors.shape}")

Loading cached product vectors...
  Product vectors loaded: (112590, 768)


### Step 13: Encode Review Sentences
This is the longest step (~4-6 hours). Progress is saved every 100K sentences.

In [8]:
REVIEW_VECTORS_FILE = os.path.join(DIRS["faiss"], "review_vectors.npy")
REVIEW_META_FILE    = os.path.join(DIRS["faiss"], "review_metadata.pkl")
CHECKPOINT_DIR      = os.path.join(DIRS["faiss"], "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if not os.path.exists(REVIEW_VECTORS_FILE):
    print("Encoding review sentences with BLAIR...")
    print(f"Total sentences: {len(df_all_sentences):,}")
    print("Saving checkpoint every 100,000 sentences.")

    all_texts = df_all_sentences["sentence"].tolist()
    CHUNK_SIZE = 100000
    chunks = []

    for chunk_start in range(0, len(all_texts), CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, len(all_texts))
        chunk_file = os.path.join(CHECKPOINT_DIR, f"chunk_{chunk_start}.npy")

        if os.path.exists(chunk_file):
            print(f"  Chunk {chunk_start:,}-{chunk_end:,}: loaded from cache")
            chunk_vectors = np.load(chunk_file)
        else:
            print(f"  Encoding chunk {chunk_start:,}-{chunk_end:,}...")
            chunk_texts   = all_texts[chunk_start:chunk_end]
            chunk_vectors = encode_texts_blair(chunk_texts, batch_size=32)
            np.save(chunk_file, chunk_vectors)
            print(f"    Checkpoint saved: {chunk_file}")

        chunks.append(chunk_vectors)

    # Concatenate all chunks
    review_vectors = np.vstack(chunks)

    # Normalize for cosine similarity
    norms = np.linalg.norm(review_vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1
    review_vectors = review_vectors / norms

    np.save(REVIEW_VECTORS_FILE, review_vectors)
    print(f"\nAll review vectors saved: {review_vectors.shape}")
else:
    print("Loading cached review vectors...")
    review_vectors = np.load(REVIEW_VECTORS_FILE)
    print(f"  Review vectors loaded: {review_vectors.shape}")

Encoding review sentences with BLAIR...
Total sentences: 1,991,664
Saving checkpoint every 100,000 sentences.
  Chunk 0-100,000: loaded from cache
  Chunk 100,000-200,000: loaded from cache
  Chunk 200,000-300,000: loaded from cache
  Chunk 300,000-400,000: loaded from cache
  Chunk 400,000-500,000: loaded from cache
  Chunk 500,000-600,000: loaded from cache
  Chunk 600,000-700,000: loaded from cache
  Chunk 700,000-800,000: loaded from cache
  Chunk 800,000-900,000: loaded from cache
  Chunk 900,000-1,000,000: loaded from cache
  Chunk 1,000,000-1,100,000: loaded from cache
  Chunk 1,100,000-1,200,000: loaded from cache
  Chunk 1,200,000-1,300,000: loaded from cache
  Chunk 1,300,000-1,400,000: loaded from cache
  Chunk 1,400,000-1,500,000: loaded from cache
  Chunk 1,500,000-1,600,000: loaded from cache
  Chunk 1,600,000-1,700,000: loaded from cache
  Chunk 1,700,000-1,800,000: loaded from cache
  Chunk 1,800,000-1,900,000: loaded from cache
  Chunk 1,900,000-1,991,664: loaded from 



### Step 14: Run CNN Sentiment Classifier on All Sentences
Label every sentence as pro/con with confidence score.

In [9]:
SENTIMENT_FILE = os.path.join(DIRS["data_proc"], "sentences_with_sentiment.parquet")

if not os.path.exists(SENTIMENT_FILE):
    print("Running CNN sentiment classifier on all sentences...")

    # --- Rebuild CNN model (same architecture as Phase 3) ---
    # CNN config (fixed setting, BLAIR embedding)
    cnn_embed_dim = 768
    best_kernels  = [3, 4, 5]
    best_filters  = 100
    best_dropout  = 0.5

    # Rebuild vocabulary
    from collections import Counter
    df_train = df_sent_labeled[df_sent_labeled["split"] == "train"]
    word_counts = Counter()
    for text in df_train["sentence"]:
        word_counts.update(text.lower().split())
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, count in word_counts.items():
        if count >= 2:
            vocab[word] = len(vocab)
    VOCAB_SIZE = len(vocab)

    # CNN model class (same as Phase 3)
    class TextCNN(nn.Module):
        def __init__(self, vocab_size, embed_dim, num_filters, kernel_sizes, dropout,
                     pretrained_embeddings=None):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
            if pretrained_embeddings is not None:
                self.embedding.weight.data.copy_(torch.tensor(pretrained_embeddings))
            self.convs   = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
            self.dropout = nn.Dropout(dropout)
            self.fc      = nn.Linear(num_filters * len(kernel_sizes), 1)

        def forward(self, x):
            x = self.embedding(x).permute(0, 2, 1)
            conv_outs = [torch.relu(conv(x)).max(dim=2).values for conv in self.convs]
            cat = self.dropout(torch.cat(conv_outs, dim=1))
            return self.fc(cat).squeeze(1)

    cnn_model = TextCNN(
        vocab_size  = VOCAB_SIZE,
        embed_dim   = cnn_embed_dim,
        num_filters = best_filters,
        kernel_sizes= best_kernels,
        dropout     = best_dropout,
    ).to(DEVICE)

    # Load best weights
    cnn_path = os.path.join(DIRS["models"], "cnn_sentiment_best.pt")
    cnn_model.load_state_dict(torch.load(cnn_path, map_location=DEVICE))
    cnn_model.eval()
    print(f"  CNN model loaded from: {cnn_path}")

    # Load confidence threshold from Phase 3
    results_json = os.path.join(DIRS["results"], "phase3_results.json")
    if os.path.exists(results_json):
        with open(results_json) as f:
            p3_results = json.load(f)
        CONFIDENCE_THRESHOLD = p3_results.get("confidence_threshold", 0.70)
    else:
        CONFIDENCE_THRESHOLD = 0.70
    print(f"  Confidence threshold: {CONFIDENCE_THRESHOLD}")

    # Run inference on all sentences
    MAX_LEN = 128
    all_probs = []
    all_texts = df_all_sentences["sentence"].tolist()

    BATCH = 512
    with torch.no_grad():
        for i in tqdm(range(0, len(all_texts), BATCH), desc="CNN Inference"):
            batch_texts = all_texts[i:i+BATCH]
            batch_indices = []
            for text in batch_texts:
                tokens  = text.lower().split()[:MAX_LEN]
                indices = [vocab.get(t, 1) for t in tokens]
                indices += [0] * (MAX_LEN - len(indices))
                batch_indices.append(indices)

            x = torch.tensor(batch_indices, dtype=torch.long).to(DEVICE)
            logits = cnn_model(x)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)

    all_probs = np.array(all_probs)

    # Assign sentiment labels
    confidence = np.maximum(all_probs, 1 - all_probs)
    sentiment  = np.where(all_probs >= 0.5, "pro", "con")

    # Sentences below threshold → "ambiguous" (stored in both pools)
    sentiment[confidence < CONFIDENCE_THRESHOLD] = "ambiguous"

    df_all_sentences["sentiment"]  = sentiment
    df_all_sentences["confidence"] = confidence
    df_all_sentences["prob_pro"]   = all_probs

    df_all_sentences.to_parquet(SENTIMENT_FILE, index=False)

    print(f"\nSentiment distribution:")
    print(df_all_sentences["sentiment"].value_counts())
    print(f"\nSaved to: {SENTIMENT_FILE}")
else:
    print("Loading cached sentiment labels...")
    df_all_sentences = pd.read_parquet(SENTIMENT_FILE)
    print(df_all_sentences["sentiment"].value_counts())

Loading cached sentiment labels...
sentiment
pro    989179
amb    731469
con    271016
Name: count, dtype: int64


### Step 15: Build FAISS HNSW Indexes

In [10]:
PRODUCT_INDEX_FILE = os.path.join(DIRS["faiss"], "product.index")
REVIEW_INDEX_FILE  = os.path.join(DIRS["faiss"], "review.index")

# --- 15a: Product vector index ---
if not os.path.exists(PRODUCT_INDEX_FILE):
    print("Building product FAISS HNSW index...")
    product_index = faiss.IndexHNSWFlat(BLAIR_DIM, 16)  # M=16
    product_index.hnsw.efConstruction = 200
    product_index.add(product_vectors.astype(np.float32))
    faiss.write_index(product_index, PRODUCT_INDEX_FILE)
    print(f"  Product index: {product_index.ntotal:,} vectors")
else:
    product_index = faiss.read_index(PRODUCT_INDEX_FILE)
    print(f"  Product index loaded: {product_index.ntotal:,} vectors")

# --- 15b: Review vector index ---
if not os.path.exists(REVIEW_INDEX_FILE):
    print("Building review FAISS HNSW index...")
    review_index = faiss.IndexHNSWFlat(BLAIR_DIM, 16)
    review_index.hnsw.efConstruction = 200
    review_index.add(review_vectors.astype(np.float32))
    faiss.write_index(review_index, REVIEW_INDEX_FILE)
    print(f"  Review index: {review_index.ntotal:,} vectors")
else:
    review_index = faiss.read_index(REVIEW_INDEX_FILE)
    print(f"  Review index loaded: {review_index.ntotal:,} vectors")

# --- 15c: Build lookup structures ---
# Map product_id -> list of sentence indices
from collections import defaultdict

product_to_sentences = defaultdict(list)
for idx, row in df_all_sentences.iterrows():
    product_to_sentences[row["parent_asin"]].append(idx)

# Save lookup
LOOKUP_FILE = os.path.join(DIRS["faiss"], "product_to_sentences.pkl")
with open(LOOKUP_FILE, "wb") as f:
    pickle.dump(dict(product_to_sentences), f)

print(f"\nProducts with reviews: {len(product_to_sentences):,}")
print(" Phase 4 complete!")

  Product index loaded: 112,590 vectors
Building review FAISS HNSW index...
  Review index: 1,991,664 vectors

Products with reviews: 111,967
 Phase 4 complete!


---
## Phase 5: BiLSTM Clarification Classifier
Detect whether a user query is generic or specific.

### Step 16: Create Synthetic Query Dataset

In [12]:
QUERY_DATASET_FILE = os.path.join(DIRS["data_proc"], "clarification_queries.csv")

if not os.path.exists(QUERY_DATASET_FILE):
    print("Creating synthetic query dataset...")

    # Product categories in beauty domain
    products = [
        "shampoo", "conditioner", "moisturizer", "sunscreen", "serum",
        "lipstick", "foundation", "mascara", "cleanser", "toner",
        "face mask", "hair oil", "body lotion", "eye cream", "perfume",
    ]

    # --- Generic query templates (lack 2+ attributes) ---
    generic_templates = [
        "recommend a {product}",
        "best {product}",
        "good {product}",
        "looking for {product}",
        "I need a {product}",
        "suggest a {product}",
        "what {product} should I buy",
        "popular {product}",
        "top rated {product}",
        "which {product} is good",
        "find me a {product}",
        "any good {product}",
        "I want a {product}",
        "help me choose a {product}",
        "what is the best {product}",
        "show me {product} options",
        "compare {product}s",
        "{product} recommendations",
        "affordable {product}",
        "luxury {product}",
    ]

    # --- Specific query templates (contain 2+ attributes) ---
    hair_types    = ["curly", "straight", "wavy", "thick", "thin", "coarse", "fine"]
    skin_types    = ["oily", "dry", "sensitive", "combination", "acne-prone", "mature"]
    concerns      = ["dryness", "frizz", "dandruff", "breakage", "color protection",
                     "anti-aging", "dark spots", "acne", "wrinkles", "hydration",
                     "oil control", "sun protection", "brightening", "pore minimizing"]
    budgets       = ["under $10", "under $15", "under $20", "under $30", "under $50"]
    ingredients   = ["sulfate-free", "paraben-free", "vegan", "organic", "natural",
                     "vitamin C", "retinol", "hyaluronic acid", "niacinamide", "salicylic acid"]

    specific_templates = [
        "{ingredient} {product} for {skin_type} skin",
        "{product} for {hair_type} hair with {concern}",
        "{product} for {concern} {budget}",
        "{ingredient} {product} for {concern}",
        "best {product} for {skin_type} skin {budget}",
        "{product} for {hair_type} {hair_type2} hair",
        "{ingredient} {product} that helps with {concern} {budget}",
        "recommend a {product} for {skin_type} skin with {concern}",
        "I need a {ingredient} {product} for my {hair_type} hair",
        "looking for {product} for {concern} and {concern2} {budget}",
    ]

    queries = []

    # Generate generic queries
    for template in generic_templates:
        for product in products:
            queries.append({
                "query": template.format(product=product),
                "label": 0  # generic
            })

    # Generate specific queries (sample to keep balanced)
    for _ in range(len(queries)):  # match generic count
        template = random.choice(specific_templates)
        product  = random.choice(products)
        query = template.format(
            product    = product,
            ingredient = random.choice(ingredients),
            skin_type  = random.choice(skin_types),
            hair_type  = random.choice(hair_types),
            hair_type2 = random.choice(hair_types),
            concern    = random.choice(concerns),
            concern2   = random.choice(concerns),
            budget     = random.choice(budgets),
        )
        queries.append({"query": query, "label": 1})  # specific

    # Shuffle
    random.shuffle(queries)

    # === Hard examples: borderline queries that are tricky to classify ===
    hard_examples = [
        # --- Looks specific but actually GENERIC (label=0) ---
        # มี 1 attribute แต่ยังขาดอีก 1-2 ตัว
        {"query": "recommend a good moisturizer for daily use", "label": 0},
        {"query": "best serum under $30", "label": 0},
        {"query": "shampoo for damaged hair", "label": 0},
        {"query": "lightweight sunscreen for summer", "label": 0},
        {"query": "foundation with good coverage", "label": 0},
        {"query": "cleanser that doesn't dry out skin", "label": 0},
        {"query": "affordable anti-aging cream", "label": 0},
        {"query": "conditioner for color-treated hair", "label": 0},
        {"query": "mascara that doesn't clump", "label": 0},
        {"query": "toner for large pores", "label": 0},
        {"query": "face mask for glowing skin", "label": 0},
        {"query": "body lotion that smells nice", "label": 0},
        {"query": "hair oil for shine", "label": 0},
        {"query": "eye cream for dark circles", "label": 0},
        {"query": "perfume that lasts long", "label": 0},
        {"query": "lip balm for dry lips", "label": 0},
        {"query": "nail polish that doesn't chip", "label": 0},
        {"query": "primer for smooth makeup", "label": 0},
        {"query": "blush that looks natural", "label": 0},
        {"query": "concealer for blemishes", "label": 0},
        {"query": "deodorant that actually works", "label": 0},
        {"query": "hair spray with strong hold", "label": 0},
        {"query": "makeup remover for waterproof makeup", "label": 0},
        {"query": "gentle shampoo for everyday use", "label": 0},
        {"query": "moisturizer with SPF", "label": 0},
        {"query": "good drugstore foundation", "label": 0},
        {"query": "serum for brighter skin", "label": 0},
        {"query": "conditioner that detangles well", "label": 0},
        {"query": "sunscreen that doesn't leave white cast", "label": 0},
        {"query": "cleanser for removing makeup", "label": 0},
        {"query": "face cream for winter", "label": 0},
        {"query": "volumizing shampoo for flat hair", "label": 0},
        {"query": "natural looking lipstick", "label": 0},
        {"query": "eye cream that really works", "label": 0},
        {"query": "body lotion for soft skin", "label": 0},
        {"query": "toner for acne", "label": 0},
        {"query": "hair mask for dry ends", "label": 0},
        {"query": "non-greasy moisturizer", "label": 0},
        {"query": "long lasting foundation for work", "label": 0},
        {"query": "hydrating serum for face", "label": 0},
        {"query": "I need something for my hair", "label": 0},
        {"query": "what's a good product for acne", "label": 0},
        {"query": "help me find a cream for wrinkles", "label": 0},
        {"query": "any suggestions for frizzy hair", "label": 0},
        {"query": "I want a nice perfume as a gift", "label": 0},
        {"query": "looking for a new skincare routine", "label": 0},
        {"query": "what do you recommend for dry skin", "label": 0},
        {"query": "I need a better shampoo", "label": 0},
        {"query": "suggest something for hair growth", "label": 0},
        {"query": "my skin is breaking out what should I use", "label": 0},

        # --- Products missing from original list (GENERIC) ---
        {"query": "recommend a lip balm", "label": 0},
        {"query": "best nail polish", "label": 0},
        {"query": "good hair spray", "label": 0},
        {"query": "looking for a primer", "label": 0},
        {"query": "suggest a blush", "label": 0},
        {"query": "best concealer", "label": 0},
        {"query": "recommend a deodorant", "label": 0},
        {"query": "good makeup remover", "label": 0},
        {"query": "I need a hair dye", "label": 0},
        {"query": "best brow pencil", "label": 0},
        {"query": "suggest a setting powder", "label": 0},
        {"query": "recommend a body wash", "label": 0},
        {"query": "good hand cream", "label": 0},
        {"query": "looking for a face mist", "label": 0},
        {"query": "best dry shampoo", "label": 0},

        # --- Looks generic but actually SPECIFIC (label=1) ---
        # สั้นแต่มีครบ 2+ attributes
        {"query": "sulfate-free shampoo for curly hair under $15", "label": 1},
        {"query": "oil-free moisturizer for acne-prone oily skin", "label": 1},
        {"query": "retinol serum for mature dry skin under $25", "label": 1},
        {"query": "vegan lipstick in nude shade under $10", "label": 1},
        {"query": "mineral sunscreen SPF 50 for sensitive face", "label": 1},
        {"query": "salicylic acid cleanser for oily acne skin", "label": 1},
        {"query": "hyaluronic acid toner for dry dehydrated skin", "label": 1},
        {"query": "paraben-free conditioner for fine colored hair", "label": 1},
        {"query": "matte foundation for oily skin under $20", "label": 1},
        {"query": "vitamin C eye cream for dark circles and wrinkles", "label": 1},
        {"query": "organic body lotion for eczema sensitive skin", "label": 1},
        {"query": "argan hair oil for thick frizzy hair under $15", "label": 1},
        {"query": "waterproof mascara for sensitive eyes under $12", "label": 1},
        {"query": "niacinamide face mask for oily large pores", "label": 1},
        {"query": "fragrance-free perfume alternative for sensitive nose", "label": 1},
        {"query": "keratin shampoo for chemically treated damaged hair", "label": 1},
        {"query": "gel moisturizer for combination acne-prone skin", "label": 1},
        {"query": "zinc sunscreen for rosacea sensitive face", "label": 1},
        {"query": "tea tree cleanser for teenage acne oily skin", "label": 1},
        {"query": "peptide serum for fine lines and firmness", "label": 1},
        {"query": "coconut-free conditioner for curly fine hair", "label": 1},
        {"query": "lightweight primer for oily skin with large pores", "label": 1},
        {"query": "natural blush for fair sensitive skin under $15", "label": 1},
        {"query": "full coverage concealer for acne scars dark skin", "label": 1},
        {"query": "aluminum-free deodorant for sensitive skin", "label": 1},
        {"query": "flexible hold hair spray for fine thin hair", "label": 1},
        {"query": "oil-based makeup remover for dry sensitive skin", "label": 1},
        {"query": "ammonia-free hair dye for gray coverage", "label": 1},
        {"query": "long-wear brow pencil for blonde sparse brows", "label": 1},
        {"query": "translucent setting powder for oily T-zone", "label": 1},
        {"query": "gentle body wash for eczema dry skin", "label": 1},
        {"query": "shea butter hand cream for cracked winter hands", "label": 1},
        {"query": "rosewater face mist for dry dehydrated skin", "label": 1},
        {"query": "volumizing dry shampoo for oily fine hair", "label": 1},
        {"query": "biotin nail polish for weak brittle nails", "label": 1},

        # --- Conversational / informal style (harder to classify) ---
        {"query": "my hair is so dry and frizzy I need a sulfate-free shampoo under 20 bucks", "label": 1},
        {"query": "what's good for oily skin that won't break the bank", "label": 0},
        {"query": "I have sensitive acne-prone skin and need a gentle vitamin C serum", "label": 1},
        {"query": "something to help with my dark spots would be nice", "label": 0},
        {"query": "need a curly hair conditioner that's paraben-free for under $15", "label": 1},
        {"query": "any eye creams actually work for wrinkles", "label": 0},
        {"query": "my oily skin hates most sunscreens, need a matte mineral SPF 50", "label": 1},
        {"query": "just want something for my face", "label": 0},
        {"query": "looking for a retinol cream for my fine lines on dry mature skin", "label": 1},
        {"query": "I keep breaking out what should I put on my face", "label": 0},
        {"query": "can you suggest a niacinamide moisturizer for oily combo skin under $25", "label": 1},
        {"query": "what's trending in skincare right now", "label": 0},
        {"query": "my daughter needs a gentle sulfate-free shampoo for her curly hair", "label": 1},
        {"query": "something smells nice for a gift", "label": 0},
        {"query": "I want an organic aloe vera gel for sunburn and sensitive skin", "label": 1},
    ]

    # Append hard examples to queries list
    queries.extend(hard_examples)
    print(f"  Added {len(hard_examples)} hard examples")

    df_queries = pd.DataFrame(queries)

    # Remove exact duplicates
    df_queries = df_queries.drop_duplicates(subset="query").reset_index(drop=True)

    df_queries.to_csv(QUERY_DATASET_FILE, index=False)
    print(f"  Total queries: {len(df_queries):,}")
    print(f"  Generic:  {(df_queries['label']==0).sum():,}")
    print(f"  Specific: {(df_queries['label']==1).sum():,}")
    print(f"  Saved to: {QUERY_DATASET_FILE}")
else:
    df_queries = pd.read_csv(QUERY_DATASET_FILE)
    print(f"Loaded {len(df_queries):,} queries")
    print(df_queries["label"].value_counts())

Loaded 712 queries
label
0    372
1    340
Name: count, dtype: int64


### Step 17: Split Data (70/15/15)

In [13]:
# Split: 70% train, 15% val, 15% test
q_train, q_temp = train_test_split(
    df_queries, test_size=0.3, random_state=SEED, stratify=df_queries["label"]
)
q_val, q_test = train_test_split(
    q_temp, test_size=0.5, random_state=SEED, stratify=q_temp["label"]
)

q_train = q_train.reset_index(drop=True)
q_val   = q_val.reset_index(drop=True)
q_test  = q_test.reset_index(drop=True)

print(f"Train: {len(q_train):,} | Val: {len(q_val):,} | Test: {len(q_test):,}")
print(f"Train label dist: {q_train['label'].value_counts().to_dict()}")

Train: 498 | Val: 107 | Test: 107
Train label dist: {0: 260, 1: 238}


### Step 18: Train BiLSTM with 5-Fold CV

In [14]:
# --- Encode queries with BLAIR ---
print("Encoding queries with BLAIR...")
train_embeddings = encode_texts_blair(q_train["query"].tolist(), batch_size=64, show_progress=False)
val_embeddings   = encode_texts_blair(q_val["query"].tolist(),   batch_size=64, show_progress=False)
test_embeddings  = encode_texts_blair(q_test["query"].tolist(),  batch_size=64, show_progress=False)
print(f"  Train: {train_embeddings.shape}, Val: {val_embeddings.shape}, Test: {test_embeddings.shape}")


# --- BiLSTM Model ---
class BiLSTMClassifier(nn.Module):
    """
    BiLSTM binary classifier for query clarification.
    Input: BLAIR embedding (768-dim) per query.
    Since each query is a single embedding, we treat it as a sequence of length 1.
    """
    def __init__(self, input_dim=768, hidden_dim=64, num_layers=1, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)  # *2 for bidirectional

    def forward(self, x):
        # x: (batch, 768) -> (batch, 1, 768)
        if x.dim() == 2:
            x = x.unsqueeze(1)
        output, (h_n, _) = self.lstm(x)
        # Concatenate forward and backward final hidden states
        hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (batch, hidden*2)
        hidden = self.dropout(hidden)
        return self.fc(hidden).squeeze(1)


# --- Dataset ---
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.tensor(embeddings, dtype=torch.float32)
        self.labels     = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):  return len(self.labels)
    def __getitem__(self, idx): return self.embeddings[idx], self.labels[idx]


# --- 5-Fold Cross-Validation ---
print("\nRunning 5-Fold CV for hyperparameter validation...")

# Combine train + val for CV
cv_embeddings = np.vstack([train_embeddings, val_embeddings])
cv_labels     = np.concatenate([q_train["label"].values, q_val["label"].values])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_f1s = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(cv_embeddings, cv_labels), 1):
    tr_data = EmbeddingDataset(cv_embeddings[tr_idx], cv_labels[tr_idx])
    va_data = EmbeddingDataset(cv_embeddings[va_idx], cv_labels[va_idx])
    tr_loader = DataLoader(tr_data, batch_size=32, shuffle=True)
    va_loader = DataLoader(va_data, batch_size=32)

    fold_model = BiLSTMClassifier(input_dim=768, hidden_dim=64, num_layers=1, dropout=0.5).to(DEVICE)
    fold_opt   = optim.Adam(fold_model.parameters(), lr=0.001)
    fold_crit  = nn.BCEWithLogitsLoss()

    best_f1 = 0
    patience = 0

    for epoch in range(50):
        fold_model.train()
        for emb, lab in tr_loader:
            emb, lab = emb.to(DEVICE), lab.to(DEVICE)
            fold_opt.zero_grad()
            loss = fold_crit(fold_model(emb), lab)
            loss.backward()
            fold_opt.step()

        # Validate
        fold_model.eval()
        preds, labels_list = [], []
        with torch.no_grad():
            for emb, lab in va_loader:
                logits = fold_model(emb.to(DEVICE))
                preds.extend((torch.sigmoid(logits) >= 0.5).long().cpu().numpy())
                labels_list.extend(lab.numpy())
        f1 = f1_score(labels_list, preds)

        if f1 > best_f1:
            best_f1 = f1
            patience = 0
        else:
            patience += 1
            if patience >= 10:
                break

    fold_f1s.append(best_f1)
    print(f"  Fold {fold}: F1 = {best_f1:.4f} (stopped at epoch {epoch+1})")

print(f"\n5-Fold CV Mean F1: {np.mean(fold_f1s):.4f} (+/- {np.std(fold_f1s):.4f})")

Encoding queries with BLAIR...


/tmp/ipykernel_476/641134279.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():  # FP16


  Train: (498, 768), Val: (107, 768), Test: (107, 768)

Running 5-Fold CV for hyperparameter validation...
  Fold 1: F1 = 0.9649 (stopped at epoch 27)
  Fold 2: F1 = 1.0000 (stopped at epoch 22)
  Fold 3: F1 = 0.9580 (stopped at epoch 19)
  Fold 4: F1 = 0.9661 (stopped at epoch 17)
  Fold 5: F1 = 0.9739 (stopped at epoch 16)

5-Fold CV Mean F1: 0.9726 (+/- 0.0146)


In [15]:
# --- Train final model on full train+val set ---
print("Training final BiLSTM on full training set...")

full_train_data   = EmbeddingDataset(cv_embeddings, cv_labels)
full_train_loader = DataLoader(full_train_data, batch_size=32, shuffle=True)

clarif_model = BiLSTMClassifier(input_dim=768, hidden_dim=64, num_layers=1, dropout=0.5).to(DEVICE)
clarif_opt   = optim.Adam(clarif_model.parameters(), lr=0.001)
clarif_crit  = nn.BCEWithLogitsLoss()

# Train for median number of epochs from CV
FINAL_EPOCHS = 30
for epoch in range(1, FINAL_EPOCHS + 1):
    clarif_model.train()
    total_loss = 0
    for emb, lab in full_train_loader:
        emb, lab = emb.to(DEVICE), lab.to(DEVICE)
        clarif_opt.zero_grad()
        loss = clarif_crit(clarif_model(emb), lab)
        loss.backward()
        clarif_opt.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"  Epoch {epoch:>3}: loss = {total_loss/len(full_train_loader):.4f}")

# Save model
CLARIF_MODEL_PATH = os.path.join(DIRS["models"], "bilstm_clarification.pt")
torch.save(clarif_model.state_dict(), CLARIF_MODEL_PATH)
print(f"Saved to: {CLARIF_MODEL_PATH}")

Training final BiLSTM on full training set...
  Epoch  10: loss = 0.0352
  Epoch  20: loss = 0.0074
  Epoch  30: loss = 0.0050
Saved to: /content/drive/MyDrive/NLP_Project/models/bilstm_clarification.pt


### Step 19: Evaluate on Test Set

In [16]:
test_data   = EmbeddingDataset(test_embeddings, q_test["label"].values)
test_loader = DataLoader(test_data, batch_size=32)

clarif_model.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for emb, lab in test_loader:
        logits = clarif_model(emb.to(DEVICE))
        test_preds.extend((torch.sigmoid(logits) >= 0.5).long().cpu().numpy())
        test_labels.extend(lab.numpy())

print("Clarification Classifier — Test Results")
print("=" * 50)
print(classification_report(
    test_labels, test_preds,
    target_names=["Generic", "Specific"],
    digits=4
))
print(f"Binary F1: {f1_score(test_labels, test_preds):.4f}")
print(" Phase 5 complete!")

Clarification Classifier — Test Results
              precision    recall  f1-score   support

     Generic     1.0000    0.9107    0.9533        56
    Specific     0.9107    1.0000    0.9533        51

    accuracy                         0.9533       107
   macro avg     0.9554    0.9554    0.9533       107
weighted avg     0.9574    0.9533    0.9533       107

Binary F1: 0.9533
 Phase 5 complete!


---
## Phase 6: BiLSTM Extractive Summarizer

### Step 20: Create Importance Labels from Helpful Votes

In [17]:
print("Creating importance labels from helpful_vote proxy...")

# For each product, label top 20% sentences by helpful_vote as important
df_summ = df_all_sentences.copy()
df_summ["helpful_vote"] = df_summ["helpful_vote"].fillna(0).astype(int)

def label_importance(group):
    """Label top 20% by helpful_vote as important (1), rest as 0."""
    threshold = group["helpful_vote"].quantile(0.80)
    group["importance"] = (group["helpful_vote"] >= threshold).astype(int)
    # If threshold is 0 (most have 0 votes), only mark those with > 0
    if threshold == 0:
        group["importance"] = (group["helpful_vote"] > 0).astype(int)
    return group

df_summ = df_summ.groupby("parent_asin", group_keys=False).apply(label_importance)

n_important = (df_summ["importance"] == 1).sum()
n_total     = len(df_summ)
print(f"Important sentences: {n_important:,} / {n_total:,} ({n_important/n_total*100:.1f}%)")

Creating importance labels from helpful_vote proxy...
Important sentences: 514,154 / 1,991,664 (25.8%)


/tmp/ipykernel_476/675896924.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_summ = df_summ.groupby("parent_asin", group_keys=False).apply(label_importance)


### Step 21: Split 80/10/10 Stratified by Product

In [18]:
# Get unique products and split at product level
unique_products = df_summ["parent_asin"].unique()

prod_train, prod_temp = train_test_split(unique_products, test_size=0.2, random_state=SEED)
prod_val, prod_test   = train_test_split(prod_temp, test_size=0.5, random_state=SEED)

df_summ["split"] = "train"
df_summ.loc[df_summ["parent_asin"].isin(prod_val),  "split"] = "val"
df_summ.loc[df_summ["parent_asin"].isin(prod_test), "split"] = "test"

for s in ["train", "val", "test"]:
    n = (df_summ["split"] == s).sum()
    print(f"  {s:>5}: {n:>10,} sentences")

  train:  1,606,871 sentences
    val:    191,901 sentences
   test:    192,892 sentences


### Step 22: Train BiLSTM Importance Scorer

In [19]:
from rouge_score import rouge_scorer

# --- Prepare BLAIR embeddings for summarizer ---
# Use pre-computed review_vectors (from Phase 4)
summ_train_mask = (df_summ["split"] == "train").values
summ_val_mask   = (df_summ["split"] == "val").values
summ_test_mask  = (df_summ["split"] == "test").values

X_train_summ = review_vectors[summ_train_mask]
y_train_summ = df_summ.loc[summ_train_mask, "importance"].values
X_val_summ   = review_vectors[summ_val_mask]
y_val_summ   = df_summ.loc[summ_val_mask, "importance"].values

train_summ_ds = EmbeddingDataset(X_train_summ, y_train_summ)
val_summ_ds   = EmbeddingDataset(X_val_summ, y_val_summ)

train_summ_loader = DataLoader(train_summ_ds, batch_size=128, shuffle=True,  num_workers=2)
val_summ_loader   = DataLoader(val_summ_ds,   batch_size=128, shuffle=False, num_workers=2)

print(f"Train: {len(train_summ_ds):,} | Val: {len(val_summ_ds):,}")


# --- BiLSTM Importance Scorer (reuse same architecture) ---
summ_model = BiLSTMClassifier(
    input_dim=768, hidden_dim=128, num_layers=1, dropout=0.5
).to(DEVICE)

summ_opt  = optim.Adam(summ_model.parameters(), lr=0.001)
summ_crit = nn.BCEWithLogitsLoss()

# --- Training loop ---
MAX_EPOCHS_SUMM = 30
PATIENCE_SUMM   = 5
SUMM_MODEL_PATH = os.path.join(DIRS["models"], "bilstm_summarizer.pt")

best_val_f1_summ = 0
patience_counter = 0

print(f"\n{'Epoch':>5} │ {'Train Loss':>10} │ {'Val F1':>8} │ Status")
print("─" * 45)

for epoch in range(1, MAX_EPOCHS_SUMM + 1):
    summ_model.train()
    train_loss = 0
    for emb, lab in train_summ_loader:
        emb, lab = emb.to(DEVICE), lab.to(DEVICE)
        summ_opt.zero_grad()
        loss = summ_crit(summ_model(emb), lab)
        loss.backward()
        summ_opt.step()
        train_loss += loss.item()
    avg_loss = train_loss / len(train_summ_loader)

    # Validate
    summ_model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for emb, lab in val_summ_loader:
            logits = summ_model(emb.to(DEVICE))
            val_preds.extend((torch.sigmoid(logits) >= 0.5).long().cpu().numpy())
            val_labels.extend(lab.numpy())
    val_f1 = f1_score(val_labels, val_preds)

    status = ""
    if val_f1 > best_val_f1_summ:
        best_val_f1_summ = val_f1
        patience_counter = 0
        torch.save(summ_model.state_dict(), SUMM_MODEL_PATH)
        status = "★ saved"
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE_SUMM:
            status = "STOP"

    print(f"{epoch:>5} │ {avg_loss:>10.4f} │ {val_f1:>8.4f} │ {status}")
    if patience_counter >= PATIENCE_SUMM:
        break

print(f"\nBest Val F1: {best_val_f1_summ:.4f}")

Train: 1,606,871 | Val: 191,901

Epoch │ Train Loss │   Val F1 │ Status
─────────────────────────────────────────────
    1 │     0.5564 │   0.0036 │ ★ saved
    2 │     0.5534 │   0.0107 │ ★ saved
    3 │     0.5523 │   0.0109 │ ★ saved
    4 │     0.5514 │   0.0159 │ ★ saved
    5 │     0.5508 │   0.0108 │ 
    6 │     0.5503 │   0.0190 │ ★ saved
    7 │     0.5498 │   0.0251 │ ★ saved
    8 │     0.5493 │   0.0292 │ ★ saved
    9 │     0.5490 │   0.0241 │ 
   10 │     0.5486 │   0.0205 │ 
   11 │     0.5481 │   0.0264 │ 
   12 │     0.5477 │   0.0226 │ 
   13 │     0.5473 │   0.0168 │ STOP

Best Val F1: 0.0292


### Step 23: Evaluate Summarizer with ROUGE

In [20]:
# Load best model
summ_model.load_state_dict(torch.load(SUMM_MODEL_PATH, map_location=DEVICE))
summ_model.eval()

# Get test products
test_products = df_summ[df_summ["split"] == "test"]["parent_asin"].unique()

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
all_rouge1, all_rouge2, all_rougeL = [], [], []

TOP_K = 5  # Select top-K sentences as summary

for product_id in tqdm(test_products[:200], desc="Evaluating ROUGE"):  # Sample 200 products
    product_mask = (df_summ["parent_asin"] == product_id) & (df_summ["split"] == "test")
    product_df   = df_summ[product_mask]

    if len(product_df) < TOP_K:
        continue

    # Get embeddings for this product's sentences
    indices = product_df.index.tolist()
    emb     = torch.tensor(review_vectors[indices], dtype=torch.float32).to(DEVICE)

    # Score sentences
    with torch.no_grad():
        scores = torch.sigmoid(summ_model(emb)).cpu().numpy()

    # Select top-K as predicted summary
    top_k_idx     = np.argsort(scores)[-TOP_K:]
    pred_summary  = " ".join(product_df.iloc[top_k_idx]["sentence"].tolist())

    # Reference: sentences actually marked as important
    ref_sentences = product_df[product_df["importance"] == 1]["sentence"].tolist()
    if len(ref_sentences) == 0:
        continue
    ref_summary = " ".join(ref_sentences)

    # Compute ROUGE
    rouge = scorer.score(ref_summary, pred_summary)
    all_rouge1.append(rouge["rouge1"].fmeasure)
    all_rouge2.append(rouge["rouge2"].fmeasure)
    all_rougeL.append(rouge["rougeL"].fmeasure)

print(f"\nSummarization Results (top-{TOP_K} sentences, {len(all_rouge1)} products):")
print(f"  ROUGE-1: {np.mean(all_rouge1):.4f}")
print(f"  ROUGE-2: {np.mean(all_rouge2):.4f}")
print(f"  ROUGE-L: {np.mean(all_rougeL):.4f}")
print(" Phase 6 complete!")

Evaluating ROUGE:   0%|          | 0/200 [00:00<?, ?it/s]


Summarization Results (top-5 sentences, 148 products):
  ROUGE-1: 0.3567
  ROUGE-2: 0.2485
  ROUGE-L: 0.2375
 Phase 6 complete!


---
## Phase 7: Ground Truth Construction + Retrieval Evaluation

### Step 24: Create Query-Product Relevance Pairs
In practice, these should be manually annotated by the team.
Below we create a semi-automated ground truth for initial evaluation.

In [21]:
GT_FILE = os.path.join(DIRS["ground_truth"], "query_product_pairs.json")

if not os.path.exists(GT_FILE):
    print("Creating ground truth query-product pairs...")
    print("Using keyword matching as proxy for manual annotation.")
    print("NOTE: Replace this with manual annotations for final evaluation.\n")

    # Sample evaluation queries
    eval_queries = [
        "sulfate-free shampoo for curly hair",
        "moisturizer for dry sensitive skin",
        "vitamin C serum for dark spots",
        "long lasting red lipstick under $20",
        "natural sunscreen for oily skin",
        "anti-aging eye cream with retinol",
        "hair oil for frizzy hair",
        "gentle cleanser for acne-prone skin",
        "hydrating face mask for dry skin",
        "organic body lotion with shea butter",
        "paraben-free conditioner for colored hair",
        "matte foundation for oily skin",
        "volumizing mascara waterproof",
        "niacinamide toner for pores",
        "leave-in conditioner for thick hair",
        "lightweight moisturizer with SPF",
        "hair growth serum for thinning hair",
        "exfoliating cleanser with salicylic acid",
        "lip balm with SPF protection",
        "hyaluronic acid serum for hydration",
        "curl defining cream for wavy hair",
        "brightening face wash for dull skin",
        "deep conditioning hair mask",
        "oil-free moisturizer for acne",
        "setting spray for makeup",
        "dry shampoo for oily hair",
        "collagen cream for wrinkles",
        "bb cream with SPF for light skin",
        "tea tree face wash for acne",
        "coconut oil hair treatment",
        "charcoal face mask for pores",
        "rosehip oil for scars",
        "micellar water makeup remover",
        "keratin shampoo for damaged hair",
        "aloe vera gel for sunburn",
        "fragrance-free lotion for eczema",
        "argan oil for dry hair",
        "pore minimizing primer",
        "anti-dandruff shampoo medicated",
        "overnight face cream anti-aging",
        "heat protectant spray for hair",
        "mineral sunscreen for face",
        "glycolic acid toner exfoliating",
        "biotin shampoo for hair loss",
        "soothing face cream for redness",
        "detangling spray for kids hair",
        "caffeine eye cream for dark circles",
        "salicylic acid spot treatment",
        "silk pillowcase for hair care",
        "jojoba oil for face moisturizing",
    ]

    # For each query, use BLAIR to find relevant products
    ground_truth = []
    product_index.hnsw.efSearch = 100  # Higher = more accurate search

    for query in eval_queries:
        q_vec = encode_texts_blair([query], batch_size=1, show_progress=False)
        q_vec = q_vec / np.linalg.norm(q_vec)

        # Search top-20 products
        distances, indices = product_index.search(q_vec.astype(np.float32), 20)

        relevant_products = []
        for idx, dist in zip(indices[0], distances[0]):
            if idx < len(product_meta["product_ids"]):
                pid   = product_meta["product_ids"][idx]
                title = product_meta["titles"][idx]
                # Simple relevance check: query keywords appear in title
                query_words = set(query.lower().split())
                title_words = set(str(title).lower().split())
                overlap = len(query_words & title_words)
                if overlap >= 2 or dist > 0.5:  # keyword match OR high similarity
                    relevant_products.append(pid)

        ground_truth.append({
            "query":    query,
            "relevant_products": relevant_products[:10],  # Cap at 10
        })

    with open(GT_FILE, "w") as f:
        json.dump(ground_truth, f, indent=2)

    print(f"Created {len(ground_truth)} query-product pairs")
    print(f"Avg relevant products per query: {np.mean([len(g['relevant_products']) for g in ground_truth]):.1f}")
    print(f"Saved to: {GT_FILE}")
    print("\nIMPORTANT: Review and manually correct these pairs for final evaluation!")
else:
    with open(GT_FILE) as f:
        ground_truth = json.load(f)
    print(f"Loaded {len(ground_truth)} ground truth pairs")

Loaded 50 ground truth pairs


### Step 25 & 26: Evaluate Retrieval (Recall@K, NDCG@K) + Baselines

In [22]:
def compute_recall_at_k(retrieved, relevant, k):
    """Proportion of relevant items found in top-K retrieved."""
    if len(relevant) == 0:
        return 0.0
    retrieved_k = set(retrieved[:k])
    return len(retrieved_k & set(relevant)) / len(relevant)


def compute_ndcg_at_k(retrieved, relevant, k):
    """NDCG@K with binary relevance."""
    if len(relevant) == 0:
        return 0.0
    rel_set = set(relevant)
    dcg = sum(1.0 / np.log2(i + 2) for i, pid in enumerate(retrieved[:k]) if pid in rel_set)
    ideal = sum(1.0 / np.log2(i + 2) for i in range(min(k, len(relevant))))
    return dcg / ideal if ideal > 0 else 0.0


def evaluate_retrieval(retrieval_func, ground_truth, ks=[5, 10, 20]):
    """Evaluate a retrieval function against ground truth."""
    results = {f"Recall@{k}": [] for k in ks}
    results.update({f"NDCG@{k}": [] for k in ks})

    for gt in ground_truth:
        if len(gt["relevant_products"]) == 0:
            continue
        retrieved = retrieval_func(gt["query"])
        for k in ks:
            results[f"Recall@{k}"].append(compute_recall_at_k(retrieved, gt["relevant_products"], k))
            results[f"NDCG@{k}"].append(compute_ndcg_at_k(retrieved, gt["relevant_products"], k))

    return {metric: np.mean(vals) for metric, vals in results.items()}


# --- Method 1: BLAIR retrieval (our system) ---
def blair_retrieve(query, top_n=20):
    q_vec = encode_texts_blair([query], batch_size=1, show_progress=False)
    q_vec = q_vec / np.linalg.norm(q_vec)
    _, indices = product_index.search(q_vec.astype(np.float32), top_n)
    return [product_meta["product_ids"][i] for i in indices[0] if i < len(product_meta["product_ids"])]


# --- Method 2: TF-IDF baseline ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Building TF-IDF baseline...")
product_texts_list = df_metadata.apply(
    lambda r: " ".join([str(r.get("title","")), " ".join(r.get("features",[])), " ".join(r.get("description",[]))]),
    axis=1
).tolist()

tfidf = TfidfVectorizer(max_features=10000, stop_words="english")
tfidf_matrix = tfidf.fit_transform(product_texts_list)

def tfidf_retrieve(query, top_n=20):
    q_vec = tfidf.transform([query])
    sims  = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(sims)[-top_n:][::-1]
    return [product_meta["product_ids"][i] for i in top_indices]


# --- Method 3: Random baseline ---
all_pids = product_meta["product_ids"]
def random_retrieve(query, top_n=20):
    return random.sample(all_pids, min(top_n, len(all_pids)))


# --- Evaluate all methods ---
print("\nEvaluating retrieval methods...\n")

blair_results  = evaluate_retrieval(blair_retrieve, ground_truth)
tfidf_results  = evaluate_retrieval(tfidf_retrieve, ground_truth)
random_results = evaluate_retrieval(random_retrieve, ground_truth)

# Display results
comparison_data = []
for metric in blair_results:
    comparison_data.append({
        "Metric": metric,
        "BLAIR":  round(blair_results[metric], 4),
        "TF-IDF": round(tfidf_results[metric], 4),
        "Random": round(random_results[metric], 4),
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

# Save
df_comparison.to_csv(os.path.join(DIRS["results"], "retrieval_comparison.csv"), index=False)
print("\n Phase 7 complete!")

Building TF-IDF baseline...

Evaluating retrieval methods...



/tmp/ipykernel_476/641134279.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():  # FP16


   Metric  BLAIR  TF-IDF  Random
 Recall@5 0.3727  0.1815     0.0
Recall@10 0.7276  0.2472     0.0
Recall@20 0.9457  0.3466     0.0
   NDCG@5 0.6878  0.3126     0.0
  NDCG@10 0.7080  0.2710     0.0
  NDCG@20 0.8194  0.3213     0.0

 Phase 7 complete!


---
## Phase 8: End-to-End Pipeline

### Step 27: Assemble Full Pipeline

In [23]:
# Predefined clarification templates
CLARIFICATION_QUESTIONS = [
    "What is your hair type or skin type? (e.g., curly, oily, dry, sensitive)",
    "What is your primary concern? (e.g., frizz, acne, anti-aging, hydration)",
    "Do you have a budget preference? (e.g., under $10, under $20, under $50)",
]


def classify_query(query):
    """Check if query is generic or specific using BiLSTM classifier."""
    clarif_model.eval()
    q_emb = encode_texts_blair([query], batch_size=1, show_progress=False)
    q_tensor = torch.tensor(q_emb, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        logit = clarif_model(q_tensor)
        prob  = torch.sigmoid(logit).item()
    return "specific" if prob >= 0.5 else "generic"


def retrieve_products(query, top_n=5):
    """Retrieve top-N relevant products using BLAIR + FAISS."""
    q_vec = encode_texts_blair([query], batch_size=1, show_progress=False)
    q_vec = q_vec / np.linalg.norm(q_vec)
    distances, indices = product_index.search(q_vec.astype(np.float32), top_n)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        if idx < len(product_meta["product_ids"]):
            results.append({
                "product_id": product_meta["product_ids"][idx],
                "title":      product_meta["titles"][idx],
                "price":      product_meta["prices"][idx],
                "avg_rating": product_meta["avg_ratings"][idx],
                "similarity": float(dist),
            })
    return results


def summarize_product(product_id, top_k=5):
    """Generate extractive pros/cons summary for a product."""
    summ_model.eval()

    # Get sentence indices for this product
    if product_id not in product_to_sentences:
        return {"pros": ["No reviews available."], "cons": []}

    sent_indices = product_to_sentences[product_id]
    product_sents = df_all_sentences.iloc[sent_indices]

    # Separate pro and con pools
    pro_mask = (product_sents["sentiment"] == "pro") | (product_sents["sentiment"] == "ambiguous")
    con_mask = (product_sents["sentiment"] == "con") | (product_sents["sentiment"] == "ambiguous")

    summaries = {"pros": [], "cons": []}

    for pool_name, mask in [("pros", pro_mask), ("cons", con_mask)]:
        pool_df = product_sents[mask]
        if len(pool_df) == 0:
            continue

        pool_indices = pool_df.index.tolist()
        emb = torch.tensor(review_vectors[pool_indices], dtype=torch.float32).to(DEVICE)

        with torch.no_grad():
            scores = torch.sigmoid(summ_model(emb)).cpu().numpy()

        # Select top-K sentences
        k = min(top_k, len(scores))
        top_idx = np.argsort(scores)[-k:]
        selected = pool_df.iloc[top_idx]
        summaries[pool_name] = selected["sentence"].tolist()

    return summaries


def run_pipeline(query, top_n_products=3, top_k_sentences=3):
    """
    Full end-to-end pipeline:
    Query -> Clarification Check -> Retrieval -> Summarization
    """
    print(f"\n{'='*70}")
    print(f"Query: \"{query}\"")
    print(f"{'='*70}")

    # Sub-task 1: Clarification check
    query_type = classify_query(query)
    print(f"\n[Sub-task 1] Query type: {query_type.upper()}")

    if query_type == "generic":
        print("\n  Clarification questions needed:")
        for i, q in enumerate(CLARIFICATION_QUESTIONS, 1):
            print(f"    Q{i}: {q}")
        print("\n  (In production, user would answer these. Proceeding with original query for demo.)")

    # Sub-task 2: Retrieval
    print(f"\n[Sub-task 2] Retrieving top-{top_n_products} products...")
    products = retrieve_products(query, top_n=top_n_products)

    # Sub-task 3: Summarization
    print(f"\n[Sub-task 3] Generating pros/cons summaries...")

    for i, product in enumerate(products, 1):
        print(f"\n  {'─'*60}")
        print(f"  Product {i}: {product['title']}")
        print(f"  Price: ${product['price']} | Rating: {product['avg_rating']} | ID: {product['product_id']}")

        summary = summarize_product(product["product_id"], top_k=top_k_sentences)

        print(f"\n  ✓ PROS:")
        if summary["pros"]:
            for j, pro in enumerate(summary["pros"], 1):
                print(f"    {j}. {pro}")
        else:
            print(f"    (No pros found in reviews)")

        print(f"\n  ✗ CONS:")
        if summary["cons"]:
            for j, con in enumerate(summary["cons"], 1):
                print(f"    {j}. {con}")
        else:
            print(f"    (No cons found in reviews)")

    return products


print("Pipeline assembled!")

Pipeline assembled!


### Step 28: Test End-to-End with Example Queries

In [24]:
# Test with generic query
run_pipeline("recommend a shampoo")


Query: "recommend a shampoo"

[Sub-task 1] Query type: GENERIC

  Clarification questions needed:
    Q1: What is your hair type or skin type? (e.g., curly, oily, dry, sensitive)
    Q2: What is your primary concern? (e.g., frizz, acne, anti-aging, hydration)
    Q3: Do you have a budget preference? (e.g., under $10, under $20, under $50)

  (In production, user would answer these. Proceeding with original query for demo.)

[Sub-task 2] Retrieving top-3 products...

[Sub-task 3] Generating pros/cons summaries...

  ────────────────────────────────────────────────────────────
  Product 1: Shampoo
  Price: $None | Rating: 3.3 | ID: B075FBFYZ4

  ✓ PROS:
    1. As mentioned it is truly organic used for infant and it worked very well
    2. Ive been using it for a week now and i like it :) it makes my scalp and hair feel soft.
    3. The softness in the end result is worth the purchase.

  ✗ CONS:
    1. It makes my hair soft but greasy and weighed down

  ────────────────────────────────

/tmp/ipykernel_476/641134279.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():  # FP16


[{'product_id': 'B075FBFYZ4',
  'title': 'Shampoo',
  'price': 'None',
  'avg_rating': 3.3,
  'similarity': 0.21789175271987915},
 {'product_id': 'B00CG89B0E',
  'title': 'This Works In Transit Shampoo 10 bottle 1oz each',
  'price': 'None',
  'avg_rating': 5.0,
  'similarity': 0.22100543975830078},
 {'product_id': 'B000ORQXZA',
  'title': 'Alpecin Aktiv-Shampoo A1-8.45 oz /250 ml - fresh from Germany',
  'price': 'None',
  'avg_rating': 4.6,
  'similarity': 0.224944606423378}]

In [25]:
# Test with specific query
run_pipeline("sulfate-free shampoo for curly dry hair under $20")


Query: "sulfate-free shampoo for curly dry hair under $20"

[Sub-task 1] Query type: SPECIFIC

[Sub-task 2] Retrieving top-3 products...

[Sub-task 3] Generating pros/cons summaries...

  ────────────────────────────────────────────────────────────
  Product 1: Wholesale Caprice Shampoo 800ml Herbal
  Price: $None | Rating: 3.7 | ID: B076B51V6D

  ✓ PROS:
    1. I will definitely be stocking up, because I worry that it will be discontinued or the formula changed.
    2. But this company was AWESOME and I am super happy with this product.
    3. The package ALWAYS comes well-packed (like Fort Knox) and I appreciate that-(not that long ago, I ordered something from a company, (liquid laundry detergent) and it wasn’t well packed and broke.

  ✗ CONS:
    1. After all these years, I'm throwing in the towel and concluding that there will never be any shampoo with a comparable herbal fragrance.
    2. This shampoo has been famous for YEARS as a knock off of the original Clairol Herbal Essen

/tmp/ipykernel_476/641134279.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():  # FP16


[{'product_id': 'B076B51V6D',
  'title': 'Wholesale Caprice Shampoo 800ml Herbal',
  'price': 'None',
  'avg_rating': 3.7,
  'similarity': 0.17556780576705933},
 {'product_id': 'B08MV711SV',
  'title': 'Curlessence Moisturizing Shampoo 12 fl. Oz.',
  'price': '11.98',
  'avg_rating': 4.1,
  'similarity': 0.17984044551849365},
 {'product_id': 'B07H8TY6BJ',
  'title': 'Alaffia Repair & Restore Conditioning Shampoo, 12 Oz',
  'price': 'None',
  'avg_rating': 3.8,
  'similarity': 0.18219667673110962}]

In [ ]:
# Test with more queries
test_queries = [
    "best moisturizer",                                   # generic
    "vitamin C serum for oily acne-prone skin",           # specific
    "recommend a sunscreen",                              # generic
    "lightweight SPF 50 sunscreen for sensitive skin",    # specific
]

for q in test_queries:
    run_pipeline(q, top_n_products=2, top_k_sentences=3)


Query: "best moisturizer"

[Sub-task 1] Query type: GENERIC

  Clarification questions needed:
    Q1: What is your hair type or skin type? (e.g., curly, oily, dry, sensitive)
    Q2: What is your primary concern? (e.g., frizz, acne, anti-aging, hydration)
    Q3: Do you have a budget preference? (e.g., under $10, under $20, under $50)

  (In production, user would answer these. Proceeding with original query for demo.)

[Sub-task 2] Retrieving top-2 products...

[Sub-task 3] Generating pros/cons summaries...

  ────────────────────────────────────────────────────────────
  Product 1: Parisian Secret Revitalizing Moisturizer
  Price: $None | Rating: 5.0 | ID: B073VFBZXR

  ✓ PROS:
    1. My skin seems to be getting better .. no dryness and hopefully will keep my face wrinkle free.
    2. Affordable unlike those promoted by celebrities .. and I'm not tied or forced to buy every 3 months nuts

  ✗ CONS:
    (No cons found in reviews)

  ──────────────────────────────────────────────────

/tmp/ipykernel_756/641134279.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():  # FP16


---
## Phase 9: Faithfulness Evaluation

### Step 29 & 30: Compute Faithfulness Score

In [26]:
print("Faithfulness Evaluation")
print("=" * 60)
print("Since this is EXTRACTIVE summarization (selecting actual review")
print("sentences), faithfulness is structurally guaranteed — every")
print("claim in the summary IS an actual review sentence.\n")
print("We verify this by checking that each summary sentence")
print("exists in the review database for the given product.\n")

# Sample 50 products for evaluation
eval_products = random.sample(
    [pid for pid in product_to_sentences if len(product_to_sentences[pid]) >= 10],
    min(50, len(product_to_sentences))
)

total_claims  = 0
grounded_claims = 0
ungrounded_examples = []

for product_id in tqdm(eval_products, desc="Checking faithfulness"):
    # Get all review sentences for this product
    sent_indices  = product_to_sentences[product_id]
    review_texts  = set(df_all_sentences.iloc[sent_indices]["sentence"].tolist())

    # Generate summary
    summary = summarize_product(product_id, top_k=5)

    # Check each claim
    for pool_name in ["pros", "cons"]:
        for sentence in summary[pool_name]:
            total_claims += 1
            if sentence in review_texts:
                grounded_claims += 1
            else:
                ungrounded_examples.append({
                    "product_id": product_id,
                    "sentence":   sentence,
                    "pool":       pool_name,
                })

faithfulness = grounded_claims / total_claims if total_claims > 0 else 0

print(f"\nResults:")
print(f"  Products evaluated:  {len(eval_products)}")
print(f"  Total claims:        {total_claims}")
print(f"  Grounded claims:     {grounded_claims}")
print(f"  Faithfulness score:  {faithfulness:.4f} ({faithfulness*100:.1f}%)")

if ungrounded_examples:
    print(f"\n  Ungrounded examples ({len(ungrounded_examples)}):")
    for ex in ungrounded_examples[:5]:
        print(f"    - [{ex['pool']}] {ex['sentence'][:80]}...")

# Save results
faith_results = {
    "num_products":      len(eval_products),
    "total_claims":      total_claims,
    "grounded_claims":   grounded_claims,
    "faithfulness_score": faithfulness,
    "ungrounded_count":  len(ungrounded_examples),
}
with open(os.path.join(DIRS["results"], "faithfulness_results.json"), "w") as f:
    json.dump(faith_results, f, indent=2)

print("\n Phase 9 complete!")

Faithfulness Evaluation
Since this is EXTRACTIVE summarization (selecting actual review
sentences), faithfulness is structurally guaranteed — every
claim in the summary IS an actual review sentence.

We verify this by checking that each summary sentence
exists in the review database for the given product.



Checking faithfulness:   0%|          | 0/50 [00:00<?, ?it/s]


Results:
  Products evaluated:  50
  Total claims:        372
  Grounded claims:     372
  Faithfulness score:  1.0000 (100.0%)

 Phase 9 complete!


---
## Phase 10: Results Summary & Demo

### Step 31: Consolidated Results

In [27]:
print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)

# --- CNN Sentiment ---
print("\n1. CNN Sentiment Classifier")
p3_path = os.path.join(DIRS["results"], "phase3_results.json")
if os.path.exists(p3_path):
    with open(p3_path) as f:
        p3 = json.load(f)
    print(f"   Test Macro F1: {p3['test_results']['test_macro_f1']:.4f}")
    print(f"   Confidence threshold: {p3['confidence_threshold']}")
print(f"   Best grid search config:")
grid_csv = os.path.join(DIRS["results"], "grid_search_results.csv")
if os.path.exists(grid_csv):
    best = pd.read_csv(grid_csv).sort_values("best_val_f1", ascending=False).iloc[0]
    print(f"     {best.to_dict()}")

# --- Clarification Classifier ---
print(f"\n2. BiLSTM Clarification Classifier")
print(f"   5-Fold CV Mean F1: {np.mean(fold_f1s):.4f} (+/- {np.std(fold_f1s):.4f})")
print(f"   Test F1: {f1_score(test_labels, test_preds):.4f}")

# --- Retrieval ---
print(f"\n3. Retrieval Performance (BLAIR vs Baselines)")
retr_csv = os.path.join(DIRS["results"], "retrieval_comparison.csv")
if os.path.exists(retr_csv):
    print(pd.read_csv(retr_csv).to_string(index=False))

# --- Summarization ---
print(f"\n4. Extractive Summarization")
print(f"   ROUGE-1: {np.mean(all_rouge1):.4f}")
print(f"   ROUGE-2: {np.mean(all_rouge2):.4f}")
print(f"   ROUGE-L: {np.mean(all_rougeL):.4f}")

# --- Faithfulness ---
print(f"\n5. Faithfulness")
print(f"   Score: {faithfulness:.4f} ({faithfulness*100:.1f}%)")

print(f"\n{'='*70}")
print("All results saved in Google Drive:")
print(f"  {DIRS['results']}/")
print(f"  {DIRS['models']}/")

FINAL RESULTS SUMMARY

1. CNN Sentiment Classifier
   Test Macro F1: 0.8198
   Confidence threshold: 0.9000000000000004
   Best grid search config:

2. BiLSTM Clarification Classifier
   5-Fold CV Mean F1: 0.9726 (+/- 0.0146)
   Test F1: 0.9533

3. Retrieval Performance (BLAIR vs Baselines)
   Metric  BLAIR  TF-IDF  Random
 Recall@5 0.3727  0.1815     0.0
Recall@10 0.7276  0.2472     0.0
Recall@20 0.9457  0.3466     0.0
   NDCG@5 0.6878  0.3126     0.0
  NDCG@10 0.7080  0.2710     0.0
  NDCG@20 0.8194  0.3213     0.0

4. Extractive Summarization
   ROUGE-1: 0.3567
   ROUGE-2: 0.2485
   ROUGE-L: 0.2375

5. Faithfulness
   Score: 1.0000 (100.0%)

All results saved in Google Drive:
  /content/drive/MyDrive/NLP_Project/results/
  /content/drive/MyDrive/NLP_Project/models/


### Step 32: Interactive Demo

In [28]:
# Interactive demo: enter your own queries
print("=" * 70)
print("INTERACTIVE DEMO")
print("Type a beauty product query and press Enter.")
print("Type 'quit' to exit.")
print("=" * 70)

while True:
    query = input("\nYour query: ").strip()
    if query.lower() in ["quit", "exit", "q"]:
        print("Demo ended.")
        break
    if len(query) == 0:
        continue
    run_pipeline(query, top_n_products=3, top_k_sentences=3)

INTERACTIVE DEMO
Type a beauty product query and press Enter.
Type 'quit' to exit.

Your query: give me the best shampoo


/tmp/ipykernel_476/641134279.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():  # FP16



Query: "give me the best shampoo"

[Sub-task 1] Query type: GENERIC

  Clarification questions needed:
    Q1: What is your hair type or skin type? (e.g., curly, oily, dry, sensitive)
    Q2: What is your primary concern? (e.g., frizz, acne, anti-aging, hydration)
    Q3: Do you have a budget preference? (e.g., under $10, under $20, under $50)

  (In production, user would answer these. Proceeding with original query for demo.)

[Sub-task 2] Retrieving top-3 products...

[Sub-task 3] Generating pros/cons summaries...

  ────────────────────────────────────────────────────────────
  Product 1: Shampoo
  Price: $None | Rating: 3.3 | ID: B075FBFYZ4

  ✓ PROS:
    1. As mentioned it is truly organic used for infant and it worked very well
    2. Ive been using it for a week now and i like it :) it makes my scalp and hair feel soft.
    3. The softness in the end result is worth the purchase.

  ✗ CONS:
    1. It makes my hair soft but greasy and weighed down

  ───────────────────────────

---
## Appendix: Export All Results for Assignment 6

In [29]:
# Export all results to a single JSON for easy report generation
final_export = {
    "project": "Review-Grounded NLP System for Beauty Products",
    "team": ["Anushka Thagle (axt5884)", "Supanut Chindawan (sxc6473)"],
    "components": {
        "sentiment_classifier": {
            "model": "TextCNN (Yoon Kim, 2014)",
            "embedding": "GloVe 300d / BLAIR 768d (grid search)",
        },
        "clarification_classifier": {
            "model": "BiLSTM (hidden=64, layers=1)",
            "cv_f1_mean": round(float(np.mean(fold_f1s)), 4),
            "cv_f1_std":  round(float(np.std(fold_f1s)), 4),
        },
        "retrieval": {
            "encoder": "BLAIR (pretrained, no fine-tuning)",
            "index":   "FAISS HNSW (M=16, efConstruction=200)",
        },
        "summarizer": {
            "model": "BiLSTM Extractive (hidden=128)",
            "rouge1": round(float(np.mean(all_rouge1)), 4),
            "rouge2": round(float(np.mean(all_rouge2)), 4),
            "rougeL": round(float(np.mean(all_rougeL)), 4),
        },
        "faithfulness": {
            "score": round(faithfulness, 4),
            "method": "extractive (structurally grounded)",
        },
    },
    "files": {
        "models": DIRS["models"],
        "results": DIRS["results"],
        "faiss_index": DIRS["faiss"],
    },
}

export_path = os.path.join(DIRS["results"], "final_results_export_r1.json")
with open(export_path, "w") as f:
    json.dump(final_export, f, indent=2)

print(f"Final results exported to: {export_path}")
print("\nAll phases complete! Ready for Assignment 6 report.")

Final results exported to: /content/drive/MyDrive/NLP_Project/results/final_results_export_r1.json

All phases complete! Ready for Assignment 6 report.
